# For RQ1: Built environment v. car ownership

**Question (RQ part 1)**
- Is mixed-use development linked to lower car ownership?
- How does it compare to density, street connectivity, transit access?

**Data**
- EPA Smart Location Database v3.0
- 1 row = 1 Census block group
- cleaned in `src/build_sld.py`

**Sample**
- metros with 1M+ people ("major cities")
- block groups with 50+ households

**Outcome**
- `pct_2plus_cars` = % of households with 2+ cars
- averages weighted by # of households

In [ ]:
import numpy as np
import pandas as pd
import altair as alt

sld = pd.read_csv("../data/processed/sld_blockgroups.csv.gz", dtype={"geoid": str, "cbsa": str})

# keep major metros + block groups with enough households
df = sld[sld["major_metro"] & sld["enough_households"]].copy()

print(len(df), "block groups")
print(df["cbsa"].nunique(), "metros")
df[["pct_2plus_cars", "mixed_use", "pop_density", "intersection_density", "transit_dist_m"]].describe().round(2)

## Step 1 · Put features on the same scale

- 4 features, 4 different units → can't compare directly
- fix: turn each into a **percentile** (0 = most car-oriented, 100 = most urban)
- split into 10 bins → avg % with 2+ cars in each bin
- steeper line = stronger link to car ownership

Transit note
- SLD has no distance if nearest stop > ¾ mile
- those all tied at "worst access" → end up as one point

In [ ]:
# transit: closer = better, so flip the sign
df["transit_access"] = -df["transit_dist_m"]
# no stop within 3/4 mile = worst possible
df.loc[df["no_stop_within_3_4mi"], "transit_access"] = -np.inf

# column name -> chart label
FEATURES = {
    "mixed_use": "Mixed use",
    "pop_density": "Population density",
    "intersection_density": "Street connectivity",
    "transit_access": "Transit access",
}
OUTCOME = "pct_2plus_cars"


def profile(data, col, label):
    """Avg % with 2+ cars (household-weighted) in each of 10 percentile bins."""
    d = data[[col, OUTCOME, "households"]].dropna().copy()

    # percentile (0-100) and bin (1-10)
    d["pct"] = d[col].rank(pct=True) * 100
    d["bin"] = np.ceil(d["pct"] / 10).clip(1, 10).astype(int)

    # weighted avg = sum(outcome * households) / sum(households)
    d["cars_x_hh"] = d[OUTCOME] * d["households"]
    g = d.groupby("bin").agg(
        x=("pct", "mean"),
        cars_x_hh=("cars_x_hh", "sum"),
        hh=("households", "sum"),
        n=("households", "size"),
    ).reset_index()
    g["y"] = (g["cars_x_hh"] / g["hh"]).round(1)
    g["feature"] = label
    return g[["feature", "bin", "x", "y", "n"]]


# one profile per feature, stacked into one table
tables = []
for col, label in FEATURES.items():
    tables.append(profile(df, col, label))
prof = pd.concat(tables, ignore_index=True)
prof.round(1)

## Step 2 · Figure 1: which feature tracks car ownership most?

- x = percentile on each feature
- y = % of households with 2+ cars
- 1 line per feature

In [ ]:
# pick colors; mixed use = red since it's our focus
colors = alt.Scale(
    domain=["Mixed use", "Population density", "Street connectivity", "Transit access"],
    range=["#d1495b", "#3a6ea5", "#2a9d8f", "#8d6cab"],
)

fig1 = alt.Chart(prof).mark_line(point=True).encode(
    x=alt.X("x:Q", title="Percentile (0 = most car-oriented, 100 = most urban)"),
    y=alt.Y("y:Q", title="Households with 2+ cars (%)", scale=alt.Scale(zero=False)),
    color=alt.Color("feature:N", title="Feature", scale=colors),
    tooltip=["feature", "y", "n"],
).properties(
    width=500,
    height=320,
    title="Which neighborhood features track car ownership most closely?",
)

fig1.save("../figures/fig1_feature_comparison.png", scale_factor=2)
fig1.save("../figures/fig1_feature_comparison.html")
fig1

**Numbers for write-up**
- `gap` = drop in % with 2+ cars, least urban bin → most urban bin
- `spearman` = rank correlation w/ % with 2+ cars (more negative = stronger)

In [ ]:
# first bin = least urban, last bin = most urban
summary = prof.sort_values("bin").groupby("feature").agg(
    least_urban=("y", "first"),
    most_urban=("y", "last"),
)
summary["gap"] = summary["least_urban"] - summary["most_urban"]

for col, label in FEATURES.items():
    summary.loc[label, "spearman"] = df[col].corr(df[OUTCOME], method="spearman")

summary.sort_values("gap", ascending=False).round(2)

## Step 3 · Figure 2: does mixed use still matter at similar densities?

- problem: dense places are also mixed, connected, near transit
- → Fig 1 can't separate mixed use from density
- fix: split into 3 density groups (low / medium / high)
- redo the profiles **inside** each group
- line still steep → feature matters even at similar density
- line flat → Fig 1 slope was mostly just density

In [ ]:
# 3 equal-size density groups
df["density_group"] = pd.qcut(df["pop_density"], 3, labels=["Low density", "Medium density", "High density"])

tables = []
for group_name in ["Low density", "Medium density", "High density"]:
    group = df[df["density_group"] == group_name]
    for col, label in FEATURES.items():
        if col == "pop_density":
            continue  # skip density itself
        t = profile(group, col, label)
        t["density_group"] = group_name
        tables.append(t)
prof2 = pd.concat(tables, ignore_index=True)

fig2 = alt.Chart(prof2).mark_line(point=True).encode(
    x=alt.X("x:Q", title="Percentile within density group"),
    y=alt.Y("y:Q", title="Households with 2+ cars (%)", scale=alt.Scale(zero=False)),
    color=alt.Color("density_group:N", title="Density group",
                    sort=["Low density", "Medium density", "High density"],
                    scale=alt.Scale(range=["#9ecae1", "#4292c6", "#08306b"])),  # light -> dark blue
    column=alt.Column("feature:N", title=None),
    tooltip=["feature", "density_group", "y", "n"],
).properties(
    width=200,
    height=240,
    title="Holding density fixed, which features still track car ownership?",
)

fig2.save("../figures/fig2_within_density.png", scale_factor=2)
fig2.save("../figures/fig2_within_density.html")
fig2

In [ ]:
# gap (least urban - most urban bin) for each feature x density group
gaps = prof2.sort_values("bin").groupby(["feature", "density_group"]).agg(
    least_urban=("y", "first"),
    most_urban=("y", "last"),
)
gaps["gap"] = gaps["least_urban"] - gaps["most_urban"]

# rows = feature, columns = density group
gaps.reset_index().pivot(index="feature", columns="density_group", values="gap")[
    ["Low density", "Medium density", "High density"]
].round(1)

## Step 4 · Add household income

- Fig 2 holds density fixed, but not income
- income matters: richer households own more cars, and income varies with density, transit, and mixed use
- → a feature could look important just because it lines up with income
- source: ACS 2014-18 median household income (table B19013), joined on `geoid` (built by `src/build_acs.py`)
  - same ACS release the SLD used for car ownership → same households, same years, same block-group boundaries

**Fallback:** if `acs_blockgroups.csv.gz` hasn't been built yet, use the SLD's `low_wage_share` (share of residents' jobs paying ≤ $1,250/month, LEHD 2017), flipped so higher = richer.
This is a much weaker income measure. Treat figures made with it as provisional.

In [ ]:
from pathlib import Path

ACS_PATH = Path("../data/processed/acs_blockgroups.csv.gz")

if ACS_PATH.exists():
    acs = pd.read_csv(ACS_PATH, dtype={"geoid": str})
    df = df.merge(acs, on="geoid", how="left")   # one-to-one on block group
    INCOME_COL = "median_hh_income"              # higher = richer
    INCOME_SOURCE = "ACS 2014-18 median household income"
else:
    df["income_proxy"] = -df["low_wage_share"]   # flip: higher = richer
    INCOME_COL = "income_proxy"
    INCOME_SOURCE = "SLD low-wage share (income proxy, ACS not yet joined)"

print("Income measure:", INCOME_SOURCE)
print(f"Block groups missing income: {df[INCOME_COL].isna().mean():.1%} (dropped from Steps 5-6)")

## Step 5 · Figure 3: density **and** income held fixed

- split into 3 income groups (equal-size, like density) → 3 × 3 = 9 groups of similar density and income
- re-rank each feature **inside** each of the 9 groups → percentile within group
- same profile as before: 10 bins → household-weighted % with 2+ cars
- rows = income group, columns = feature, lines = density group (same blue ramp as Fig 2)
- read it like Fig 2: steep line = feature still tracks car ownership among similar-density, similar-income neighborhoods

Design choices
- small multiples with a **shared y-axis** → slopes compare directly across all 27 lines (position on a common scale)
- density group is ordinal → single-hue light-to-dark blue, matching Fig 2
- each line averages ~1,000+ block groups per bin, so bin means are precise. The real uncertainty is the spread *within* bins (see Next steps)

In [ ]:
DENSITY_GROUPS = ["Low density", "Medium density", "High density"]
INCOME_GROUPS = ["Lower income", "Middle income", "Higher income"]

# 3 equal-size income groups (block groups with no income value get NaN and are left out)
df["income_group"] = pd.qcut(df[INCOME_COL], 3, labels=INCOME_GROUPS)

tables = []
for inc in INCOME_GROUPS:
    for dens in DENSITY_GROUPS:
        cell = df[(df["income_group"] == inc) & (df["density_group"] == dens)]
        for col, label in FEATURES.items():
            if col == "pop_density":
                continue  # density is held fixed here, so skip it
            t = profile(cell, col, label)   # percentiles are re-ranked inside this cell
            t["income_group"] = inc
            t["density_group"] = dens
            tables.append(t)
prof3 = pd.concat(tables, ignore_index=True)

# block groups in each density x income cell
pd.crosstab(df["income_group"], df["density_group"])

In [ ]:
fig3 = alt.Chart(prof3).mark_line(point=alt.OverlayMarkDef(size=20)).encode(
    x=alt.X("x:Q", title="Percentile within group", scale=alt.Scale(domain=[0, 100])),
    y=alt.Y("y:Q", title="2+ cars (%)", scale=alt.Scale(zero=False)),
    color=alt.Color("density_group:N", title="Density group", sort=DENSITY_GROUPS,
                    scale=alt.Scale(range=["#9ecae1", "#4292c6", "#08306b"])),
    row=alt.Row("income_group:N", title=None, sort=INCOME_GROUPS,
                header=alt.Header(labelAngle=0, labelAlign="left", labelFontWeight="bold")),
    column=alt.Column("feature:N", title=None, sort=["Mixed use", "Street connectivity", "Transit access"],
                      header=alt.Header(labelFontWeight="bold", labelFontSize=13)),
    tooltip=["feature", "income_group", "density_group", "bin", "y", "n"],
).properties(
    width=190,
    height=130,
    title=alt.Title(
        "Holding density and income fixed, which features still track car ownership?",
        subtitle=[f"Households with 2+ cars (%), by percentile of each feature within each density × income group. "
                  f"{df['income_group'].notna().sum():,} block groups in 54 metros of 1M+ people.",
                  f"Income: {INCOME_SOURCE}. Source: EPA Smart Location Database v3.0."],
    ),
)

fig3.save("../figures/fig3_within_density_income.png", scale_factor=2)
fig3.save("../figures/fig3_within_density_income.html")
fig3

**How to read Fig 3**

- each panel compares neighborhoods with similar density **and** similar income, so a slope can't come from those two differences
- the lines sit at different heights because denser and richer groups own different numbers of cars on average; what matters is the **slope inside each line**
- Fig 4 (next) boils the 27 lines down to one number per feature

## Step 6 · Figure 4: how much of each link survives the controls?

- one number per feature: **gap** = % with 2+ cars in the least-urban bin minus the most-urban bin
- computed 3 ways:
  - **No controls**: Fig 1
  - **Density held fixed**: Fig 2, averaged over the 3 density groups
  - **Density + income held fixed**: Fig 3, averaged over the 9 groups
- population density can't be held fixed against itself → its last point holds **income** fixed only
- a gap that shrinks a lot as controls are added was partly riding on density or income
- caveat for mixed use: its curve is an inverted U, so end-to-end gaps understate its mid-range effect
- a negative gap means the most-urban tenth owns *more* cars than the least-urban tenth

Design choices
- position along one common x-axis (the most accurate encoding for comparing magnitudes)
- control level is ordinal → light-to-dark ramp; a grey rule connects each feature's points to show how far it moves
- features sorted by raw gap, so the ranking reads top to bottom

In [ ]:
def gap_table(p, group_cols):
    """Gap (least-urban bin minus most-urban bin) per feature, averaged over the groups in group_cols."""
    g = p.sort_values("bin").groupby(["feature"] + group_cols)["y"].agg(["first", "last"])
    g["gap"] = g["first"] - g["last"]
    return g.groupby("feature")["gap"].mean()

# density can only be held fixed against income: re-rank density inside each income group
tables = []
for inc in INCOME_GROUPS:
    t = profile(df[df["income_group"] == inc], "pop_density", "Population density")
    t["income_group"] = inc
    tables.append(t)
prof_density_income = pd.concat(tables, ignore_index=True)

LEVELS = ["No controls", "Density held fixed", "Density + income held fixed"]
gaps = pd.concat([
    gap_table(prof, []).rename(LEVELS[0]),
    gap_table(prof2, ["density_group"]).rename(LEVELS[1]),
    pd.concat([gap_table(prof3, ["income_group", "density_group"]),
               gap_table(prof_density_income, ["income_group"])]).rename(LEVELS[2]),
], axis=1)
gaps.round(1)

In [ ]:
gaps_long = gaps.reset_index().melt(id_vars="feature", var_name="controls", value_name="gap").dropna()
feature_order = gaps[LEVELS[0]].sort_values(ascending=False).index.tolist()

base = alt.Chart(gaps_long).encode(
    y=alt.Y("feature:N", title=None, sort=feature_order, axis=alt.Axis(labelFontSize=12)),
)
rules = base.mark_rule(color="#bbbbbb", strokeWidth=2).encode(
    x=alt.X("min(gap):Q"), x2="max(gap):Q",
)
dots = base.mark_circle(size=140, opacity=1).encode(
    x=alt.X("gap:Q", title="Drop in % of households with 2+ cars, least → most urban tenth (points)",
            scale=alt.Scale(domain=[-5, 60])),
    color=alt.Color("controls:N", title=None, sort=LEVELS,
                    scale=alt.Scale(domain=LEVELS, range=["#c6dbef", "#4292c6", "#08306b"]),
                    legend=alt.Legend(orient="top")),
    tooltip=["feature", "controls", alt.Tooltip("gap:Q", format=".1f")],
)
zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color="black", strokeWidth=1).encode(x="x:Q")
fig4 = (zero + rules + dots).properties(
    width=480,
    height=170,
    title=alt.Title(
        "How much of each feature's link to car ownership survives the controls?",
        subtitle=[f"Gaps averaged over density (3) and density × income (9) groups. Population density's last point holds income only.",
                  f"Income: {INCOME_SOURCE}. Source: EPA Smart Location Database v3.0."],
    ),
)

fig4.save("../figures/fig4_gap_summary.png", scale_factor=2)
fig4.save("../figures/fig4_gap_summary.html")
fig4

**What Fig 4 shows** (ACS 2014-18 income)

| Feature | No controls | Density fixed | Density + income fixed |
| --- | --- | --- | --- |
| Population density | 51.8 | n/a | 46.4 |
| Transit access | 36.1 | 23.2 | 18.6 |
| Mixed use | −2.0 | 6.7 | 10.4 |
| Street connectivity | 31.2 | 10.9 | 8.7 |

- **Density dominates**: even among similar-income neighborhoods, the densest tenth has ~46 points fewer households with 2+ cars than the least dense
- **Mixed use is the only feature that gets *stronger* with controls**: −2 → 6.7 → 10.4
  - with no controls, the least-mixed block groups include dense, residential-only apartment areas that already own few cars, which hides the mixed-use effect
  - once neighborhoods are compared at similar density and income, more mixed use goes with ~10 points fewer 2+ car households
- **Street connectivity mostly rode on density**: 31 → 9 points. Well-connected grids are usually dense, so the raw link overstated it
- **Transit access keeps the most** of the three non-density features (18.6), but about half its raw gap was density and income
- answer to our question: mixed use has a **real but modest** link to lower car ownership. It's comparable to street connectivity and about half of transit access, and well below density. Our hypothesis that mixed use matters *most* is not supported

Limits
- these are associations, not causal effects. Households who don't want to drive may choose mixed-use neighborhoods (self-selection)
- terciles are coarse controls. Some density and income variation remains inside each group
- block groups are small. A residential-only block group next to a shopping street counts as "unmixed" even though shops are a short walk away (modifiable areal unit problem)

## Step 7 · Anomaly check: dense but poorly connected block groups

- Fig 2 oddity: in the **high-density** group, the least-connected tenth has unusually *low* car ownership
- guess: large-block public or older apartment housing → lower income, fewer cars
- test: compare income and car ownership in that tenth vs the rest of the high-density group
- second guess: it's mostly one metro (New York), whose large blocks and car-free households pull the tenth down → check which metros it comes from

In [ ]:
hd = df[df["density_group"] == "High density"].copy()
hd["conn_pct"] = hd["intersection_density"].rank(pct=True) * 100
hd["tenth"] = np.where(hd["conn_pct"] <= 10, "Least-connected tenth", "Other 90%")

# show income in its original units (dollars for ACS, low-wage share for the proxy)
income_shown = "median_hh_income" if INCOME_COL == "median_hh_income" else "low_wage_share"

hd.groupby("tenth").agg(
    block_groups=("geoid", "size"),
    pct_2plus_cars=("pct_2plus_cars", "mean"),
    pct_0_cars=("pct_0_cars", "mean"),
    median_income_measure=(income_shown, "median"),
    share_in_nyc_metro=("cbsa_name", lambda s: s.str.startswith("New York").mean()),
    share_lower_income_group=("income_group", lambda s: (s == "Lower income").mean()),
).round(2)

In [ ]:
# which metros do those least-connected, high-density block groups sit in?
hd[hd["tenth"] == "Least-connected tenth"]["cbsa_name"].value_counts().head(8)

**What Step 7 shows**

| High-density block groups | Least-connected tenth | Other 90% |
| --- | --- | --- |
| Households with 2+ cars | 27% | 42% |
| Households with 0 cars | 36% | 19% |
| Median household income | $53,100 | $57,386 |
| Share in lower-income third | 47% | 42% |
| Share in New York metro | 47% | 21% |

- **income explains only a little**: the least-connected tenth is somewhat poorer (about $4,000 lower median income), not enough for a 15-point drop in 2+ car ownership
- **New York explains most of it**: nearly half these block groups are in the NYC metro, more than twice the rate in the rest of the group. NYC combines very low car ownership with large blocks (e.g. tower-in-the-park housing on superblocks), so "least connected" here largely means "in New York"
- implication: connectivity's dip at the low end of the high-density line is a metro-composition effect, not evidence that poor connectivity reduces car ownership
- possible robustness check: rank features **within each metro** instead of nationally, so one metro can't dominate a group

## Next steps

- **Within-metro check:** re-rank features inside each metro (Step 7 shows New York can dominate a group)
- **Commuting:** swap the outcome for ACS drive-alone / transit / walk shares (already in `acs_blockgroups.csv.gz`)
- **Within-bin spread:** add interquartile bands to Fig 3 so readers see the variation behind each mean
- **CO₂ (RQ part 2):** join Vulcan on-road emissions (`src/build_vulcan.py`) → plot per resident + per km² against the same percentiles